# Load Data : ensuring proper aggregation and labels

In [ ]:
import pandas as pd
import warnings
import numpy as np
warnings.filterwarnings("ignore")
df_deep=pd.read_csv("/kaggle/input/deep-features/deep_features.csv")

In [ ]:
df_handcrafted=pd.read_csv("/kaggle/input/handcrafted-features/Handcrafted_Features_flair.csv")
df_namecsv=pd.read_csv('/kaggle/input/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training/name_mapping.csv')


In [ ]:
#Aggregating the deep features by id to avoid data leakage
import pandas as pd

feature_cols = [col for col in df_deep.columns if col not in ['id', 'label']]
df_deep_agg = df_deep.groupby('id')[feature_cols].mean().reset_index()

print("Aggregated deep shape:", df_deep_agg.shape)

df_fused = pd.merge(df_deep_agg, df_handcrafted, on='id', how='inner')

print("Fused feature matrix shape:", df_fused.shape) 

X_fusion = df_fused.drop(columns=['id','label'])
y_fusion= df_fused['label']

In [ ]:
df_handcrafted

In [ ]:
y_handcrafted=df_handcrafted['label']

In [ ]:
df_namecsv['id']=df_namecsv['BraTS_2019_subject_ID']
df_namecsv['label']=df_namecsv['Grade']

In [ ]:
df_label=df_namecsv[['id','label']]

In [ ]:
df_deep_agg=pd.merge(df_deep_agg,df_label,on='id',how='inner')
df_deep_agg

In [ ]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y_deep=le.fit_transform(df_deep_agg['label'])

In [ ]:
# Check if data is properly merged
print("Data alignment check:")
print(f"Handcrafted y: {np.bincount(y_handcrafted)}")  # [76, 259]
print(f"Deep y: {np.bincount(y_deep)}")                # Should be [76, 259]
print(f"Fused y: {np.bincount(y_fusion)}")             # [76, 259]

# If Deep is reversed, fix it:
if np.bincount(y_deep)[0]!= np.bincount(y_handcrafted)[0]:
    print("\n⚠️ Deep labels are REVERSED! Flipping...")
    y_deep_fixed = 1 - y_deep  # Flip 0↔1
    y_deep = y_deep_fixed
    print(f"Fixed Deep y: {np.bincount(y_deep)}")  # Should now be [76, 259]

In [ ]:
X_deep=df_deep_agg.drop(columns=['id','label'])
X_handcrafted=df_handcrafted.drop(columns=['id','label'])

# Defining Models and Metrics

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (make_scorer, precision_score, recall_score, f1_score, 
                             roc_auc_score, confusion_matrix, cohen_kappa_score)
from sklearn.model_selection import StratifiedKFold, cross_validate
import math

In [ ]:
#defining classifiers with pipelining 
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

models= {
    "XGBoost": Pipeline([
        ("scaler", StandardScaler()), 
        ("clf", XGBClassifier(n_estimators=50, learning_rate=0.03, random_state=42, eval_metric='logloss'))
    ]),
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=10000, solver='lbfgs', C=1.0, random_state=42, n_jobs=-1))
    ]),
    "Naive Bayes": Pipeline([
        ("scaler", StandardScaler()), 
        ("clf", GaussianNB())
    ]),
    "Linear SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel='linear', probability=True, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("scaler", StandardScaler()), 
        ("clf", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
    ]),
}

In [ ]:
#for PCA model definition pipelining
from sklearn.decomposition import PCA

pca_n_components=189

models_PCA = {
    "XGBoost": Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=pca_n_components, random_state=42)),
        ("clf", XGBClassifier(
            n_estimators=50,
            learning_rate=0.03,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            gamma=1,
            scale_pos_weight=1,
            random_state=42,
            use_label_encoder=False,
            eval_metric='logloss'
        ))
    ]),
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=pca_n_components, random_state=42)),
        ("clf", LogisticRegression(
            max_iter=10000,
            solver='lbfgs',
            penalty='l2',
            C=1.0,
            random_state=42,
            n_jobs=-1
        ))
    ]),
    "Naive Bayes": Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=pca_n_components, random_state=42)),
        ("clf", GaussianNB())
    ]),
    "Linear SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=pca_n_components, random_state=42)),
        ("clf", SVC(
            kernel='linear', 
            probability=True,
            random_state=42
        ))
    ]),
    "Random Forest": Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=pca_n_components, random_state=42)),
        ("clf", RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            bootstrap=True,
            random_state=42,
            n_jobs=-1
        ))
    ]),
}

# MAIN PERFORMANCE TABLE

In [ ]:
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

In [ ]:
def compute_classwise_auc(model, X, y, cv):
    auc_class0, auc_class1 = [], []
    for train_idx, test_idx in cv.split(X, y):
        model.fit(X[train_idx], y[train_idx])
        probs = model.predict_proba(X[test_idx])
        y_true = y[test_idx]
        auc0 = roc_auc_score(y_true, probs[:,0])
        auc1 = roc_auc_score(y_true, probs[:,1])
        auc_class0.append(auc0)
        auc_class1.append(auc1)
    return np.mean(auc_class0), np.std(auc_class0), np.mean(auc_class1), np.std(auc_class1)

In [ ]:
def main_performance_table(X, y, models, runs=5, n_splits=5):
    scoring = {
        'accuracy': 'accuracy',
        'precision_macro': make_scorer(precision_score, average='macro', zero_division=0),
        'precision_weighted': make_scorer(precision_score, average='weighted', zero_division=0),
        'recall_macro': make_scorer(recall_score, average='macro', zero_division=0),
        'recall_weighted': make_scorer(recall_score, average='weighted', zero_division=0),
        'f1_macro': make_scorer(f1_score, average='macro', zero_division=0),
        'f1_weighted': make_scorer(f1_score, average='weighted', zero_division=0),
         'roc_auc': 'roc_auc',
        'specificity': make_scorer(specificity_score),
        'cohen_kappa': make_scorer(cohen_kappa_score)
    }

    accum_results = {name: {metric: [] for metric in scoring.keys()} for name in models.keys()}
    auc_results = {name: {"auc0": [], "auc1": []} for name in models.keys()}
    
    for run in range(runs):
        print(f"Run {run+1}/{runs}")
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42+run)
        for name, model in models.items():
            cv_results = cross_validate(model, X, y, cv=cv, scoring=scoring, return_estimator=True)
            for metric in scoring.keys():
                accum_results[name][metric].extend(cv_results[f'test_{metric}'])
            
            auc_class0, _, auc_class1, _ = compute_classwise_auc(model, X, y, cv)
            auc_results[name]["auc0"].append(auc_class0)
            auc_results[name]["auc1"].append(auc_class1)
    
    # Now compute mean ± std per model for each metric and AUC
    summary = []
    import math
    for name in models.keys():
        row = {"Model": name}
        for metric in scoring.keys():
            metric_vals = accum_results[name][metric]
            row[metric] = f"{np.mean(metric_vals):.3f} ± {np.std(metric_vals):.3f}"
        auc0_avg = np.mean(auc_results[name]["auc0"])
        auc0_std = np.std(auc_results[name]["auc0"])
        auc1_avg = np.mean(auc_results[name]["auc1"])
        auc1_std = np.std(auc_results[name]["auc1"])
        # Average AUC
        auc_avg = (auc0_avg + auc1_avg) / 2
        auc_std_avg = math.sqrt((auc0_std**2 + auc1_std**2) / 2)
        row["AUC_class_0"] = f"{auc0_avg:.3f} ± {auc0_std:.3f}"
        row["AUC_class_1"] = f"{auc1_avg:.3f} ± {auc1_std:.3f}"
        row["AUC_average"] = f"{auc_avg:.3f} ± {auc_std_avg:.3f}"
        summary.append(row)
    
    df_results = pd.DataFrame(summary)
    print(df_results)

In [ ]:
#deep features only
main_performance_table(X_deep.values,y_deep,models,runs=5,n_splits=5)

#handcrafted features only
main_performance_table(X_handcrafted.values,y_handcrafted,models, runs=5, n_splits=5)

#fusion only
main_performance_table(X_fusion.values,y_fusion,models,runs=5,n_splits=5)

#pca-optimised features
main_performance_table(X_fusion.values,y_fusion,models_PCA,runs=5,n_splits=5)

# PAIRED T_TEST

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

def statistical_significance_study(feature_sets, y, n_splits=5, runs=5, target="Fusion+PCA"):
    """
    Evaluates models across repeated Stratified K-Fold splits and runs
    paired t-tests comparing `target` against the other feature sets.
    """
    #Initialize data structure to store fold-level accuracies
    fold_results = {name: [] for name in feature_sets.keys()}
    
    print(f"--- Starting Study: {runs} runs of {n_splits}-fold CV ---")

    #CV loop
    for run in range(runs):
        print(f"Processing Run {run+1}/{runs}...")
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42 + run)
        
        for name, X in feature_sets.items():
            X_values = X.values if hasattr(X, 'values') else X
            
            for train_idx, test_idx in skf.split(X_values, y):
                X_train, X_test = X_values[train_idx], X_values[test_idx]
                y_train, y_test = y[train_idx], y[test_idx]

                if name == "Fusion+PCA":
                    model = models_PCA["Logistic Regression"]
                elif name == "Handcrafted":
                    model = models["Random Forest"]
                else:
                    model = models["Logistic Regression"]

                model.fit(X_train, y_train)
                
                acc = accuracy_score(y_test, model.predict(X_test))
                fold_results[name].append(acc)
                
    report_data = []
    for name, scores in fold_results.items():
        mean_acc = np.mean(scores)
        std_acc = np.std(scores)
        report_data.append({
            "Feature Set": name,
            "Accuracy": f"{mean_acc:.4f} ± {std_acc:.4f}",
            "Mean": mean_acc,
            "Std": std_acc
        })

    #Statistical Significance Testing (Paired t-test)
    print(f"\nStatistical Significance ({target} vs. Others)")
    if target in fold_results:
        target_scores = fold_results[target]
        for other, other_scores in fold_results.items():
            if other == target:
                continue
            
            t_stat, p_val = stats.ttest_rel(target_scores, other_scores)
            sig_label = "Significant" if p_val < 0.05 else "Not Significant"
            print(f"{target} vs {other}: t-stat = {t_stat:.4f}, p-value = {p_val:.6e} ({sig_label})")

    return pd.DataFrame(report_data)

feature_input = {
    "Handcrafted": X_handcrafted,
    "Deep-only": X_deep,
    "Fusion+PCA": X_fusion
}

results_df = statistical_significance_study(feature_input, y_fusion)
print("\nFinal Results Table:")
print(results_df[["Feature Set", "Accuracy"]].to_string(index=False))

# CONFUSION MATRIX for top models

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

def plot_conf_matrices(y_true, y_pred, title_prefix):
    cm_raw = confusion_matrix(y_true, y_pred)
    cm_norm = confusion_matrix(y_true, y_pred, normalize='true')  # row normalization

    # Raw counts
    disp_raw = ConfusionMatrixDisplay(confusion_matrix=cm_raw)
    disp_raw.plot(cmap='Blues')
    plt.title(f'{title_prefix} - Raw')
    plt.show()

    # Normalized (%)
    disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm)
    disp_norm.plot(cmap='Blues')
    plt.title(f'{title_prefix} - Normalized (%)')
    plt.show()

feature_set={
    'Handcrafted': 
    {
        'X' : X_handcrafted,
        'y' : y_handcrafted
    },
    'Deep':
    {
        'X': X_deep,
        'y':y_deep
    },
    'Fusion':
    {
        'X': X_fusion,
        'y':y_fusion
    },
    'PCA':
    {
        'X':X_fusion,
        'y':y_fusion
    }
}

model_sets = {
    'Handcrafted': 'Random Forest',
    'Deep': 'Naive Bayes',
    'Fusion':'Logistic Regression',
    'PCA': 'Logistic Regression'
}

In [ ]:
from sklearn.model_selection import train_test_split
feature_sets = ['Handcrafted','Deep','Fusion','PCA']
metrics = ['AUC']

# Usage example:
for feat in feature_sets:
    for metric in metrics:
        model_name = model_sets[feat]
        print(f"For {feat} features, top model : {model_name}")
        X=feature_set[feat]['X']
        y= feature_set[feat]['y']
        X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
        print('X_test length : ',len(X_test))
        print('y_test length : ',len(y_test))
        model=models[model_name]
        if feat =='PCA' :
            model=models_PCA[model_name]
        model.fit(X_train,y_train)
        y_pred=model.predict(X_test)
        plot_conf_matrices(y_test, y_pred, title_prefix=f"{feat}-{model_name}")

# COMPUTE AND EFFICIENCY METRICS

In [ ]:
import time
import sys
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
def get_real_param_count(model):
    
    # 1. Unwrap Pipeline if present
    if isinstance(model, Pipeline):
        # The actual classifier is usually the last step
        estimator = model.steps[-1][1]
    else:
        estimator = model

    try:
        # Linear Models (Logistic Regression, Linear SVM)
        if hasattr(estimator, 'coef_'):
            # (n_classes * n_features) + n_classes (intercepts)
            return estimator.coef_.size + estimator.intercept_.size
        
        # Naive Bayes
        if hasattr(estimator, 'theta_'): 
            # (n_classes * n_features) means + (n_classes * n_features) variances
            return estimator.theta_.size + estimator.var_.size
            
        # Random Forest
        if hasattr(estimator, 'estimators_'):
            # Sum of nodes in all decision trees
            return sum([tree.tree_.node_count for tree in estimator.estimators_])
            
        #XGBoost
        if hasattr(estimator, 'get_booster'):
            booster = estimator.get_booster()
            dump = booster.get_dump()
            # Count split lines to approximate nodes
            return sum([len(tree.split('\n')) for tree in dump])
        
        #SVM 
        if hasattr(estimator, 'support_vectors_'):
            # Number of support vectors * n_features
            return estimator.support_vectors_.size
            
    except Exception as e:
        pass
        
    return "N/A"

def compute_metrics(models, X_train, y_train, X_test):
    results_list = []
    
    for model_name, model in models.items():
        print(f"\n=== Measuring: {model_name} ===")
        
        start_time = time.time()
        model.fit(X_train, y_train)
        train_time = time.time() - start_time
        print(f"Training Time: {train_time:.4f} s")
        
        inference_times = []
        num_samples = min(20, len(X_test)) 
        
        try: _ = model.predict(X_test[0:1])
        except: pass

        for i in range(num_samples):
            sample = X_test[i:i+1]
            t0 = time.time()
            _ = model.predict(sample)
            t1 = time.time()
            inference_times.append(t1 - t0)
            
        avg_inference = np.mean(inference_times) * 1000 
        std_inference = np.std(inference_times) * 1000 
        print(f"Inference Time: {avg_inference:.4f} ms")
        
        params = get_real_param_count(model)
        
        try:
            p = pickle.dumps(model)
            memory_mb = len(p) / (1024 * 1024)
        except:
            memory_mb = "N/A"
            
        results_list.append({
            'Model': model_name,
            'Train Time (s)': f"{train_time:.4f}",
            'Inference (ms)': f"{avg_inference:.4f} ± {std_inference:.4f}",
            'Model Size (MB)': f"{memory_mb:.4f}" if isinstance(memory_mb, float) else memory_mb,
            'Parameters': params
        })
        
    return pd.DataFrame(results_list)

In [ ]:
#handcrafted features only
X_train, X_test, y_train, y_test = train_test_split( X_handcrafted, y_handcrafted, test_size=0.2, stratify=y_handcrafted, random_state=42)
metric_df=compute_metrics(models,X_train,y_train,X_test)
print(metric_df)

#deep features only
X_train, X_test, y_train, y_test = train_test_split( X_deep, y_deep, test_size=0.2, stratify=y_handcrafted, random_state=42)
metric_df=compute_metrics(models,X_train,y_train,X_test)
print(metric_df)

#fused features only
X_train, X_test, y_train, y_test = train_test_split( X_fusion, y_fusion, test_size=0.2, stratify=y_handcrafted, random_state=42)
metric_df=compute_metrics(models,X_train,y_train,X_test)
print(metric_df)

# fused+PCA features only
X_train, X_test, y_train, y_test = train_test_split( X_fusion, y_fusion, test_size=0.2, stratify=y_handcrafted, random_state=42)
metric_df=compute_metrics(models_PCA,X_train,y_train,X_test)
print(metric_df)

# MCNEMAR'S TEST FOR PAIRED CLASSIFICATION

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


df_fusion_all = pd.merge(df_deep_agg, df_handcrafted, on='id', how='inner', suffixes=('_deep', '_hand'))

print(f"Total Aligned Samples: {len(df_fusion_all)}")


master_ids = df_fusion_all['id'].values

df_deep_aligned = df_deep_agg.set_index('id').reindex(master_ids).reset_index()

df_hand_aligned = df_handcrafted.set_index('id').reindex(master_ids).reset_index()


#SAFETY CHECK & GROUND TRUTH SELECTION

# We default to the Fusion/Deep label as the "Master Label"
y_master = df_fusion_all['label_y'] if 'label_y' in df_fusion_all.columns else df_fusion_all['label_deep']

# Optional: Check for corruption in Handcrafted labels, but DO NOT CRASH
y_hand_check = df_hand_aligned['label']
mismatches = np.sum(y_master.values != y_hand_check.values)

if mismatches > 0:
    print(f"⚠️ WARNING: Found {mismatches} label mismatches in Handcrafted file.")
    print("   -> Ignoring Handcrafted labels and using Deep/Fusion labels as Ground Truth.")
else:
    print("✅ SUCCESS: Deep and Handcrafted labels match perfectly.")

# PREPARE X MATRICES (Drop IDs and Labels)
cols_to_drop_std = ['image_id', 'id', 'label']
cols_to_drop_fusion = ['image_id', 'id', 'label_deep', 'label_hand', 'label_x', 'label_y']

X_deep_final = df_deep_aligned.drop(columns=[c for c in cols_to_drop_std if c in df_deep_aligned.columns])
X_hand_final = df_hand_aligned.drop(columns=[c for c in cols_to_drop_std if c in df_hand_aligned.columns])
X_fusion_final = df_fusion_all.drop(columns=[c for c in cols_to_drop_fusion if c in df_fusion_all.columns])

In [ ]:
from sklearn.model_selection import train_test_split
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np


train_idx, test_idx = train_test_split(
    np.arange(len(y_master)), 
    test_size=0.2, 
    stratify=y_master, 
    random_state=42
)

# Handcrafted
X_train_hand = X_hand_final.iloc[train_idx]
X_test_hand  = X_hand_final.iloc[test_idx]

# Deep
X_train_deep = X_deep_final.iloc[train_idx]
X_test_deep  = X_deep_final.iloc[test_idx]

# Fused
X_train_fusion = X_fusion_final.iloc[train_idx]
X_test_fusion  = X_fusion_final.iloc[test_idx]
y_train = y_master.values[train_idx]
y_test  = y_master.values[test_idx]

print(f"Data Split Complete. Training on {len(y_train)} samples, Testing on {len(y_test)}.")


# Model 1: Fused
clf_fused = models["Logistic Regression"]
clf_fused.fit(X_train_fusion, y_train)
y_pred_fused = clf_fused.predict(X_test_fusion)

# Model 2: Deep
clf_deep = models["Logistic Regression"]
clf_deep.fit(X_train_deep, y_train)
y_pred_deep = clf_deep.predict(X_test_deep)

# Model 3: Handcrafted
clf_hand = models["Random Forest"]
clf_hand.fit(X_train_hand, y_train)
y_pred_hand = clf_hand.predict(X_test_hand)

# Model 4: PCA
clf_PCA = models_PCA["Logistic Regression"]
clf_PCA.fit(X_train_fusion, y_train)
y_pred_PCA = clf_PCA.predict(X_test_fusion)


def run_mcnemar(y_true, pred_A, pred_B, name_A, name_B):
    b = np.sum((pred_A == y_true) & (pred_B != y_true))
    c = np.sum((pred_A != y_true) & (pred_B == y_true))
    
    table = [[0, b], [c, 0]]
    result = mcnemar(table, exact=True)
    
    sig = "Significant" if result.pvalue < 0.05 else "Not Significant"
    print(f"McNemar {name_A} vs {name_B}: Statistic={result.statistic:.2f}, p-value={result.pvalue:.4f} ({sig})")

print("\n--- McNemar's Test Results ---")
# Fused vs Deep
run_mcnemar(y_test, y_pred_fused, y_pred_deep, "Fused", "Deep")

# Fused vs Handcrafted
run_mcnemar(y_test, y_pred_fused, y_pred_hand, "Fused", "Handcrafted")

# Fused+PCA vs Deep
run_mcnemar(y_test, y_pred_PCA, y_pred_deep, "Fused+PCA", "Deep")

# Fused+PCA vs Handcrafted
run_mcnemar(y_test, y_pred_PCA, y_pred_hand, "Fused+PCA", "Handcrafted")

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar

def get_contingency_table(y_true, pred_A, pred_B, name_A="Model A", name_B="Model B"):
    y_true = np.asarray(y_true)
    pred_A = np.asarray(pred_A)
    pred_B = np.asarray(pred_B)

    # a: Both correct
    a = np.sum((pred_A == y_true) & (pred_B == y_true))
    # b: A correct, B incorrect
    b = np.sum((pred_A == y_true) & (pred_B != y_true))
    # c: A incorrect, B correct
    c = np.sum((pred_A != y_true) & (pred_B == y_true))
    # d: Both incorrect
    d = np.sum((pred_A != y_true) & (pred_B != y_true))


    table_df = pd.DataFrame(
        [[a, b], [c, d]],
        index=[f"{name_A} Correct", f"{name_A} Incorrect"],
        columns=[f"{name_B} Correct", f"{name_B} Incorrect"]
    )
    result = mcnemar([[a, b], [c, d]], exact=True)

    print(f" Contingency Table: {name_A} vs. {name_B} (Total N = {len(y_true)})")
    print(table_df)
    print(f"Discordant Pairs: b = {b} ({name_A} only), c = {c} ({name_B} only)")
    print(f"McNemar Statistic: {result.statistic:.2f}, p-value: {result.pvalue:.4f}")

    return table_df, result

#Fused vs. Deep
table_fused_deep, res_fused_deep = get_contingency_table(
    y_test, y_pred_fused, y_pred_deep, 
    name_A="Fused", name_B="Deep"
)

#Fused vs. Handcrafted
table_fused_hand, res_fused_hand = get_contingency_table(
    y_test, y_pred_fused, y_pred_hand, 
    name_A="Fused", name_B="Handcrafted"
)

#Fused+PCA vs. Deep
table_pca_deep, res_pca_deep = get_contingency_table(
    y_test, y_pred_PCA, y_pred_deep, 
    name_A="Fused+PCA", name_B="Deep"
)

#Fused+PCA vs. Handcrafted
table_pca_hand, res_pca_hand = get_contingency_table(
    y_test, y_pred_PCA, y_pred_hand, 
    name_A="Fused+PCA", name_B="Handcrafted"
)

#  T_SNE or UMAP visualization of features

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X_handcrafted_scaled = StandardScaler().fit_transform(X_handcrafted)
X_deep_scaled = StandardScaler().fit_transform(X_deep)
X_fusion_scaled = StandardScaler().fit_transform(X_fusion)

tsne_hand = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(X_handcrafted_scaled)
tsne_deep = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(X_deep_scaled)
tsne_fusion = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(X_fusion_scaled)

X_pca= PCA(n_components=189, random_state=42).fit_transform(X_fusion_scaled)
tsne_pca=TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(X_pca)


embeddings = [
    (tsne_deep, y_deep, "t-SNE: Deep"),
    (tsne_hand, y_handcrafted, "t-SNE: Handcrafted"),
    (tsne_fusion, y_fusion, "t-SNE: Fused"),
    (tsne_pca, y_fusion, "t-SNE: Fused+PCA")
]

print("tsne-map")
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i, (emb, y_feat, title) in enumerate(embeddings):
    ax = axes.flatten()[i]
    scatter = ax.scatter(emb[:,0], emb[:,1], c=y_feat, cmap='coolwarm', s=30, alpha=0.8)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    for value, label, color in zip([0, 1], ['LGG','HGG'], ['blue','red']):
        ax.scatter([], [], c=color, label=label)
    ax.legend()
plt.suptitle('2D Embeddings Colored by Class (LGG vs HGG) for Deep, Handcrafted, and Fused Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('embeddings_separability.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import umap

X_handcrafted_scaled = StandardScaler().fit_transform(X_handcrafted)
X_deep_scaled = StandardScaler().fit_transform(X_deep)
X_fusion_scaled = StandardScaler().fit_transform(X_fusion)


umap_hand = umap.UMAP(n_components=2, random_state=42, min_dist=0.1).fit_transform(X_handcrafted_scaled)
umap_deep = umap.UMAP(n_components=2, random_state=42, min_dist=0.1).fit_transform(X_deep_scaled)
umap_fusion = umap.UMAP(n_components=2, random_state=42, min_dist=0.1).fit_transform(X_fusion_scaled)

X_pca= PCA(n_components=189, random_state=42).fit_transform(X_fusion_scaled)
umap_pca = umap.UMAP(n_components=2, random_state=42, min_dist=0.1).fit_transform(X_pca)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
embeddings = [
    (umap_deep, y_deep, "UMAP: Deep"),
    (umap_hand, y_handcrafted, "UMAP: Handcrafted"),
    (umap_fusion, y_fusion, "UMAP: Fused"),
    (umap_pca, y_fusion, "UMAP: Fused+PCA")
]

print("umap")
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i, (emb, y_feat, title) in enumerate(embeddings):
    ax = axes.flatten()[i]
    scatter = ax.scatter(emb[:,0], emb[:,1], c=y_feat, cmap='coolwarm', s=30, alpha=0.8)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    for value, label, color in zip([0, 1], ['LGG','HGG'], ['blue','red']):
        ax.scatter([], [], c=color, label=label)
    ax.legend()
plt.suptitle('2D Embeddings Colored by Class (LGG vs HGG) for Deep, Handcrafted, and Fused Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('embeddings_separability.png', dpi=300, bbox_inches='tight')
plt.show()